In [ ]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.9 MB/s eta 0:00:00


In [ ]:
import PyPDF2
print("PyPDF2 is installed and ready to use!")

PyPDF2 is installed and ready to use!


In [ ]:
import nltk

# Download necessary resources
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
import nltk
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root

True

In [ ]:
!pip install gradio

In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 69.5 MB/s eta 0:00:00


In [ ]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 78.9 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [ ]:
!pip install pdfplumber
!pip install gradio

In [ ]:
import gradio as gr
from PyPDF2 import PdfReader
import re
from nltk.corpus import wordnet
import nltk
nltk.download('wordnet')

# ✅ Get conceptual definitions from WordNet
def get_meanings(word):
    synsets = wordnet.synsets(word)
    definitions = [syn.definition() for syn in synsets if syn.definition()]
    return definitions if definitions else ["❌ No definitions found."]

# ✅ Enhanced logic to track position-wise occurrences and map meanings
def search_word_in_pdf(file_path, word):
    reader = PdfReader(file_path)
    full_text = ""

    for page in reader.pages:
        text = page.extract_text()
        if text:
            full_text += text + " "

    sentences = re.split(r'(?<=[.?!])\s+', full_text)
    word = word.strip()
    position_data = []  # Holds (position, sentence) where word appears

    position_counter = 1
    for sentence in sentences:
        count_in_sentence = sentence.lower().count(word.lower())
        for _ in range(count_in_sentence):
            position_data.append((position_counter, sentence.strip()))
            position_counter += 1

    word_count = len(position_data)
    base_meanings = get_meanings(word)
    if base_meanings and base_meanings[0] != "❌ No definitions found.":
        meanings = [base_meanings[i % len(base_meanings)] for i in range(word_count)]
    else:
        meanings = ["❌ No definitions found."] * word_count

    # Create table: [Position, Sentence, Meaning]
    result_table = [[pos, sent, meaning] for (pos, sent), meaning in zip(position_data, meanings)]

    summary = f"✅ The word '{word}' appeared {word_count} times in the PDF."
    return summary, result_table

# ✅ Gradio UI
interface = gr.Interface(
    fn=search_word_in_pdf,
    inputs=[
        gr.File(label="📄 Upload PDF File", elem_id="upload-pdf", file_count="single", type="filepath"),
        gr.Textbox(label="🔍 Enter Word to Search", elem_id="word-search", placeholder="Type a word you want to find...")
    ],
    outputs=[
        gr.Textbox(label="📜 Summary", elem_id="summary", interactive=False, placeholder="Results will appear here..."),
        gr.Dataframe(headers=["Position", "Sentence", "Meaning"], elem_id="positionwise")
    ],
    title="🚀 StorySense: An NLP Approach for Word Frequency and Contextual Meaning Analysis",
    description="Upload a PDF, search for a word, and see each occurrence with its contextual sentence and meaning from WordNet.",
    css="""
        @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;600&display=swap');

        body {
            font-family: 'Poppins', sans-serif;
            background-color: #f8fafc;
            margin: 0;
            padding: 0;
            color: #1e293b;
        }

        .gradio-container {
            background: linear-gradient(to bottom right, #eef2f7, #ffffff);
            border-radius: 20px;
            box-shadow: 0 12px 24px rgba(0, 0, 0, 0.1);
            padding: 40px;
            max-width: 1200px;
            margin: 40px auto;
        }

        h1 {
            font-size: 2.5rem;
            font-weight: 700;
            color: #0f172a;
            margin-bottom: 10px;
            text-align: center;
        }

        p {
            font-size: 1.2rem;
            text-align: center;
            margin-bottom: 40px;
            color: #475569;
        }

        #upload-pdf, #word-search {
            padding: 16px;
            border: 2px solid #38bdf8;
            border-radius: 10px;
            background-color: #f0f9ff;
            font-size: 1rem;
        }

        #summary {
            background-color: #e0f7fa;
            padding: 18px;
            border-radius: 10px;
            font-size: 1.2rem;
            font-weight: 500;
            color: #0d9488;
        }

        .gr-button {
            background-color: #38bdf8;
            color: #ffffff;
            font-weight: 600;
            font-size: 1.1rem;
            padding: 14px 24px;
            border-radius: 10px;
            border: none;
            transition: all 0.3s ease;
        }

        .gr-button:hover {
            background-color: #0284c7;
        }

        .gr-output {
            background-color: #f1f5f9;
            border-radius: 12px;
            padding: 20px;
            font-size: 1rem;
        }

        .gri-dataframe {
            background-color: #ffffff;
            border: 1px solid #cbd5e1;
            border-radius: 12px;
            box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.05);
            overflow: hidden;
            margin-bottom: 20px;
        }

        .gri-dataframe thead {
            background-color: #38bdf8;
            color: #ffffff;
            font-weight: bold;
        }

        .gri-dataframe td {
            padding: 12px;
            text-align: center;
            color: #334155;
        }
    """
)

# ✅ Launch
interface.launch(share=True)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
/usr/local/lib/python3.12/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  super().__init__(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f23ead3eed30d1b23c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from collections import Counter
import pdfplumber
import json
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from google.colab import files
from IPython.display import display, HTML
import unicodedata
import re

nltk.download('punkt')

# ===============================================================
# 📂 UPLOAD DICTIONARY + PDF
# ===============================================================

display(HTML("<h2 style='color:#2E7D32;'>📘 Upload your Marathi Dictionary (JSON)</h2>"))
uploaded_dict = files.upload()
dict_path = list(uploaded_dict.keys())[0]

with open(dict_path, "r", encoding="utf-8") as f:
    marathi_dict = json.load(f)

display(HTML("<h2 style='color:#2E7D32;'>📄 Upload your PDF file</h2>"))
uploaded_pdf = files.upload()
pdf_path = list(uploaded_pdf.keys())[0]


# ===============================================================
# Utility Functions
# ===============================================================

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + " "
    return unicodedata.normalize('NFC', text)


def normalize_text(s):
    return unicodedata.normalize("NFC", s.strip().lower())


def tokenize_marathi(s):
    s = normalize_text(s)
    tokens = word_tokenize(s)
    if not tokens:
        tokens = re.findall(r'\w+', s)
    return tokens


def sentences_with_word(text, word):
    sents = sent_tokenize(text)
    word = normalize_text(word)
    return [s for s in sents if re.search(rf'\b{re.escape(word)}\b', normalize_text(s))]


def count_word(text, word):
    word = normalize_text(word)
    return len(re.findall(r'\b{}\b'.format(re.escape(word)), normalize_text(text)))


# ===============================================================
# ⭐ HYBRID LESK (ALL 5 MEANINGS, NO SCORES)
# ===============================================================

def hybrid_lesk_all(sentence, word, marathi_dict):
    word = normalize_text(word)

    if word not in marathi_dict:
        return []

    meanings = marathi_dict[word]

    # return all meanings AS THEY ARE (no scoring output shown)
    clean_list = []
    for entry in meanings:
        meaning = entry.get("meaning", "") if isinstance(entry, dict) else entry
        clean_list.append(meaning)

    return clean_list


def highlight(sentence, word):
    return re.sub(rf'\b({re.escape(word)})\b',
                  r"<b style='color:#C62828;'>\1</b>",
                  sentence)


# ===============================================================
# MAIN FUNCTION
# ===============================================================

def main():
    text = extract_text_from_pdf(pdf_path)

    display(HTML("<h3 style='color:#1E88E5;'>🔍 Enter Marathi Word:</h3>"))
    target = input("Enter Marathi Word: ").strip()

    count = count_word(text, target)
    sents = sentences_with_word(text, target)

    # ---------------- HEADER -----------------
    display(HTML(f"""
    <div style="
        background:#E8F5E9;
        padding:15px;
        margin-top:20px;
        border-radius:8px;
        border-left:6px solid #43A047;">
        <h3 style="color:#2E7D32;">✔ निकाल</h3>
        <p><b>शब्द:</b> <span style="color:#C62828;">{target}</span></p>
        <p><b>PDF मध्ये आलेली संख्या:</b> <span style="color:#00695C;">{count}</span></p>
    </div>
    """))

    if not sents:
        display(HTML("<p style='color:red;'>❌ शब्द PDF मध्ये नाही.</p>"))
        return

    # ---------------- SENTENCES + MEANINGS -----------------
    display(HTML("<h3 style='color:#283593;'>📌 PDF मधील वाक्ये आणि त्यांचे सर्व ५ अर्थ (Context Based)</h3>"))

    for i, s in enumerate(sents, start=1):
        all_meanings = hybrid_lesk_all(s, target, marathi_dict)
        highlighted = highlight(s, target)

        html_block = f"""
        <div style="
            background:#F1F8E9;
            border-left:5px solid #8BC34A;
            padding:12px;
            margin-bottom:12px;
            border-radius:6px;">
            <p><b>{i}. वाक्य:</b> {highlighted}</p>
            <h4 style='color:#00695C;'>🔽 सर्व ५ संदर्भातील अर्थ</h4>
        """

        for idx, meaning in enumerate(all_meanings, start=1):
            html_block += f"""
                <div style="margin-left:20px; margin-bottom:10px;">
                    <b>{idx}. अर्थ:</b> {meaning}
                    <hr>
                </div>
            """

        html_block += "</div>"
        display(HTML(html_block))


main()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Saving Marathi_Json.txt to Marathi_Json (2).txt


Saving chhotya_goshti_bn_jajoo.pdf to chhotya_goshti_bn_jajoo (1).pdf


Enter Marathi Word: घर


In [ ]:
# ===============================================================
# GRADIO UI + YOUR ORIGINAL HYBRID LESK 5-MEANING LOGIC (NO SCORES)
# ===============================================================

from collections import Counter
import pdfplumber
import json
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
import unicodedata
import re
import gradio as gr

nltk.download('punkt')

# ===============================================================
# Utility Functions
# ===============================================================

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + " "
    return unicodedata.normalize('NFC', text)

def normalize_text(s):
    return unicodedata.normalize("NFC", s.strip().lower())

def tokenize_marathi(s):
    s = normalize_text(s)
    tokens = word_tokenize(s)
    if not tokens:
        tokens = re.findall(r'\w+', s)
    return tokens

def sentences_with_word(text, word):
    sents = sent_tokenize(text)
    word = normalize_text(word)
    return [s for s in sents if re.search(rf'\b{re.escape(word)}\b', normalize_text(s))]

def count_word(text, word):
    word = normalize_text(word)
    return len(re.findall(r'\b{}\b'.format(re.escape(word)), normalize_text(text)))

# ===============================================================
# ⭐ HYBRID LESK (ALL 5 MEANINGS, NO SCORES)
# ===============================================================

def hybrid_lesk_all(sentence, word, marathi_dict):
    word = normalize_text(word)
    if word not in marathi_dict:
        return []
    meanings = marathi_dict[word]

    clean_list = []
    for entry in meanings:
        meaning = entry.get("meaning", "") if isinstance(entry, dict) else entry
        clean_list.append(meaning)

    return clean_list

# ===============================================================
# MAIN LOGIC for UI
# ===============================================================

def analyze_word(pdf_file, dict_file, target_word):
    if pdf_file is None or dict_file is None:
        return "❌ कृपया PDF आणि JSON दोन्ही फाईल अपलोड करा.", []

    # Load dictionary
    try:
        with open(dict_file.name, "r", encoding="utf-8") as f:
            marathi_dict = json.load(f)
    except:
        return "❌ JSON dictionary वाचताना त्रुटी.", []

    # Extract PDF text
    text = extract_text_from_pdf(pdf_file.name)

    target_word = normalize_text(target_word)
    word_count = count_word(text, target_word)
    sents = sentences_with_word(text, target_word)

    if not sents:
        return f"'{target_word}' हा शब्द PDF मध्ये नाही.", []

    # Prepare table
    rows = []
    for i, s in enumerate(sents, start=1):
        meanings = hybrid_lesk_all(s, target_word, marathi_dict)
        meaning_join = "\n".join([f"{idx}. {m}" for idx, m in enumerate(meanings, start=1)])
        rows.append([i, s, meaning_join])

    summary = f"✅ '{target_word}' हा शब्द PDF मध्ये {word_count} वेळा आला आहे."
    return summary, rows

# ===============================================================
# ⭐ GRADIO UI
# ===============================================================

interface = gr.Interface(
    fn=analyze_word,
    inputs=[
        gr.File(label="📄 Upload Marathi PDF File", file_count="single", type="filepath"),
        gr.File(label="📘 Upload Marathi Dictionary (JSON)", file_count="single", type="filepath"),
        gr.Textbox(label="🔍 Enter Marathi Word to Search", placeholder="इथे शब्द टाइप करा...")
    ],
    outputs=[
        gr.Textbox(label="📜 Summary", interactive=False),
        gr.Dataframe(headers=["क्रमांक", "वाक्य", "सर्व ५ अर्थ"], wrap=True)
    ],
    title="📚 Marathi Word Frequency and Contextual Meaning Analyzerr",
    theme="soft"
)

interface.launch(share=True)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/usr/local/lib/python3.12/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4c12fc5d93ebad8b7a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 50.7 MB/s eta 0:00:00


In [ ]:
from collections import Counter
import fitz  # PyMuPDF
import json
import nltk
from nltk.tokenize import word_tokenize
from google.colab import files
from IPython.display import display, HTML
import unicodedata
import re

nltk.download('punkt')

# ===============================================================
# 📂 UPLOAD DICTIONARY + PDF
# ===============================================================

display(HTML("<h2 style='color:#2E7D32;'>📘 Upload your Marathi Dictionary (JSON)</h2>"))
uploaded_dict = files.upload()
dict_path = list(uploaded_dict.keys())[0]

with open(dict_path, "r", encoding="utf-8") as f:
    marathi_dict = json.load(f)

display(HTML("<h2 style='color:#2E7D32;'>📄 Upload your PDF file</h2>"))
uploaded_pdf = files.upload()
pdf_path = list(uploaded_pdf.keys())[0]

# ===============================================================
# Utility Functions
# ===============================================================

def extract_text_from_pdf(pdf_path):
    """
    Extract text from PDF using PyMuPDF.
    Better than pdfplumber for Marathi Unicode PDFs.
    """
    doc = fitz.open(pdf_path)

    text = ""

    for page in doc:
        page_text = page.get_text("text")
        if page_text:
            text += "\n" + page_text

    doc.close()

    text = unicodedata.normalize("NFC", text)

    # Clean spaces
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n+', '\n', text)

    return text


def normalize_text(s):
    return unicodedata.normalize("NFC", s.strip().lower())


def tokenize_marathi(s):
    s = normalize_text(s)

    try:
        tokens = word_tokenize(s)
    except:
        tokens = re.findall(r'[\u0900-\u097F]+', s)

    return tokens


def split_marathi_sentences(text):
    """
    Split Marathi sentences using Marathi punctuation.
    """

    text = re.sub(r'\s+', ' ', text)

    sentences = re.split(r'(?<=[।.!?])\s+', text)

    sentences = [s.strip() for s in sentences if s.strip()]

    return sentences


def sentences_with_word(text, word):

    word = normalize_text(word)

    sentences = split_marathi_sentences(text)

    matched = []

    for sentence in sentences:

        if re.search(rf'\b{re.escape(word)}\b', normalize_text(sentence)):
            matched.append(sentence)

    return matched


def count_word(text, word):

    word = normalize_text(word)

    return len(re.findall(rf'\b{re.escape(word)}\b', normalize_text(text)))


# ===============================================================
# ⭐ HYBRID LESK (ALL 5 MEANINGS)
# ===============================================================

def hybrid_lesk_all(sentence, word, marathi_dict):

    word = normalize_text(word)

    if word not in marathi_dict:
        return []

    meanings = marathi_dict[word]

    clean = []

    for entry in meanings:

        if isinstance(entry, dict):
            clean.append(entry.get("meaning", ""))

        else:
            clean.append(entry)

    return clean


def highlight(sentence, word):

    return re.sub(
        rf'({re.escape(word)})',
        r"<b style='color:#C62828;'>\1</b>",
        sentence,
        flags=re.IGNORECASE
    )


# ===============================================================
# MAIN FUNCTION
# ===============================================================

def main():

    text = extract_text_from_pdf(pdf_path)

    display(HTML("<h3 style='color:#1E88E5;'>🔍 Enter Marathi Word:</h3>"))

    target = input("Enter Marathi Word: ").strip()

    count = count_word(text, target)

    sents = sentences_with_word(text, target)

    display(HTML(f"""
    <div style="
        background:#E8F5E9;
        padding:15px;
        margin-top:20px;
        border-radius:8px;
        border-left:6px solid #43A047;">

        <h3 style="color:#2E7D32;">✔ निकाल</h3>

        <p><b>शब्द:</b>
        <span style="color:#C62828;">{target}</span></p>

        <p><b>PDF मध्ये आलेली संख्या:</b>
        <span style="color:#00695C;">{count}</span></p>

    </div>
    """))

    if not sents:

        display(HTML("<h3 style='color:red;'>❌ शब्द PDF मध्ये आढळला नाही.</h3>"))

        return

    display(HTML("<h3 style='color:#283593;'>📌 PDF मधील वाक्ये आणि त्यांचे सर्व ५ अर्थ</h3>"))

    for i, sentence in enumerate(sents, start=1):

        meanings = hybrid_lesk_all(sentence, target, marathi_dict)

        sentence = highlight(sentence, target)

        html = f"""
        <div style="
            background:#F1F8E9;
            border-left:6px solid #8BC34A;
            padding:15px;
            margin-bottom:15px;
            border-radius:8px;">

        <p><b>{i}. वाक्य:</b><br><br>{sentence}</p>

        <h4 style="color:#00695C;">
        🔽 सर्व ५ संदर्भातील अर्थ
        </h4>
        """

        if meanings:

            for j, meaning in enumerate(meanings, start=1):

                html += f"""
                <div style="margin-left:20px;">
                <b>{j}.</b> {meaning}
                <hr>
                </div>
                """

        else:

            html += "<p style='color:red;'>अर्थ उपलब्ध नाही.</p>"

        html += "</div>"

        display(HTML(html))


main()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Saving Marathi_Json.txt to Marathi_Json (1).txt


Saving chhotya_goshti_bn_jajoo.pdf to chhotya_goshti_bn_jajoo.pdf


Enter Marathi Word: घर 
